# Traditional RAG

In [1]:
from anthropic import Anthropic
anthropic_client = Anthropic()

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [3]:
from minsearch import AppendableIndex

In [4]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [5]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [6]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, tell the user you can't find the answer.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [7]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [15]:
messages = []

messages.append({
    "role": "user",
    "content": user_prompt
})

response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    system=RAG_INSTRUCTIONS,
    messages=messages
)

In [16]:
print(response.content)

[TextBlock(citations=None, text="I can't find information about creating a dashboard in Evidently in the provided documentation context. \n\nThe context I have access to covers how to use Evidently's LLM judge functionality and create datasets for evaluation, but it doesn't include documentation about dashboard creation.\n\nTo get accurate information about creating dashboards in Evidently, I'd recommend:\n1. Checking the official Evidently documentation website\n2. Looking at the Evidently Platform documentation\n3. Visiting the Evidently GitHub repository for examples\n\nIs there something else about Evidently's evaluation or monitoring features I can help you with?", type='text')]


# Making it Agentic

In [18]:
# Search tool schema

instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, tell the user you can't find the answer.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [69]:
messages = [{"role": "user", "content": user_prompt}]
model="claude-haiku-4-5"
tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}]

response = anthropic_client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=tools
)

response.usage.input_tokens

23878

In [72]:
tool_call = response.content[0]

In [73]:
messages.append(tool_call)

In [45]:
search_results = search(query='create dashboard in Evidently')

In [94]:
#search_results[:1]

In [80]:
import json

for block in response.content:
    if block.type == "tool_use":
        call_output = {
            "type": "tool_result",
            "tool_use_id": block.id, 
            "content": json.dumps(search_results)
        }

In [82]:
messages = [{"role": "user", "content": user_prompt}]
model="claude-haiku-4-5"
tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}]

response = anthropic_client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=tools,
)

response.usage.input_tokens

22289

In [87]:
answer = "\n".join(
    block.text 
    for block in response.content 
    if block.type == "text"
)

In [88]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [91]:
response = anthropic_client.messages.parse(
        model=model,
        max_tokens=1000,
        messages=messages,
        tools=tools,
        output_format=RAGResponse
    )
print(response.usage.input_tokens)

23930


In [93]:
rag_response = response.parsed_output